# 01 · Exploración de datos y auditoría de alineación (Overture Maps + Esri / Massachusetts Roads)

**Proyecto:** ¿Sirven las features de DINOv2 fuera de ImageNet? Extracción de redes viales frente a CNN (Grupo 5, Deep Learning).

Este notebook construye y entiende los dos conjuntos de datos del proyecto y **cuantifica** qué tan bien se alinean las máscaras viales con la imagen satelital:

| Parte | Contenido |
|---|---|
| **A** | Dataset propio: segmentos viales de Overture Maps (solo vectores) + imágenes Esri World Imagery (zoom 17) → parches RGB 512×512 con máscaras binarias rasterizadas, EDA y split espacial por bloques. |
| **B** | Auditoría de alineación Overture vs Esri: desplazamiento (dx, dy) estimado por bordes y por similitud de color, curva IoU vs desplazamiento, proxy de completitud y plantilla de QA manual. |
| **C** | Massachusetts Roads (benchmark de referencia): EDA y la **misma** auditoría de alineación para comparar. |

**Por qué importa la alineación:** Overture no trae imágenes. Las máscaras salen de vectores y las imágenes de otra fuente; si ambas capas no coinciden con precisión de píxel, el modelo aprende de etiquetas ruidosas y el IoU máximo alcanzable baja. Aquí se mide, no se asume.

## Cómo ejecutar en Kaggle
1. Crear un notebook nuevo e importar este archivo (File → Import notebook).
2. **Settings → Accelerator:** GPU T4 (este notebook no usa GPU, pero se mantiene la misma configuración que el notebook 02). **Internet: On** (obligatorio: se descarga Overture y los tiles de Esri).
3. **Add Input** → agregar el dataset *Massachusetts Roads Dataset* (por ejemplo `balraj98/massachusetts-roads-dataset`). Solo es necesario para la Parte C; las Partes A y B funcionan sin él.
4. Ejecutar todas las celdas en orden. Las salidas quedan en `/kaggle/working/` (`overture_lima/`, `results/`, `figures/`).
5. Al final, guardar la versión (Save Version → Save & Run All) y convertir la salida en un Dataset (ver última celda).

## 0. Instalación de dependencias
Se instala solo lo que Kaggle no trae por defecto: `overturemaps` (cliente de descarga) y, si faltara, `rasterio`. El resto (geopandas, shapely, scipy, PIL, pandas) ya viene preinstalado.

In [ ]:
!pip install -q overturemaps

import importlib.util
import subprocess
import sys

# Instalación protegida: solo si el módulo no está disponible.
for pkg, mod in [("rasterio", "rasterio"), ("geopandas", "geopandas"), ("pyarrow", "pyarrow"),
                 ("shapely", "shapely"), ("tqdm", "tqdm")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Dependencias listas.")

## 1. CONFIG (única celda de configuración)
Todos los parámetros y rutas están aquí. `BBOX` es configurable: el valor por defecto cubre distritos centrales de Lima (formato `lon_min, lat_min, lon_max, lat_max`, EPSG:4326). `MAX_PATCHES = None` usa todos los parches del bbox.

In [ ]:
# ----------------------------- Dataset propio (Overture + Esri) -----------------------------
BBOX = (-77.08, -12.15, -76.95, -12.03)   # (lon_min, lat_min, lon_max, lat_max). Configurable: Lima central.
ZOOM = 17                                  # nivel de zoom de los tiles Esri (~1.17 m/px a la latitud de Lima)
TILE = 256                                 # tamaño de tile XYZ en px
PATCH = 512                                # parche = 2x2 tiles de 256 px
MAX_PATCHES = 600                          # None = todos los parches del bbox
ESRI_URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
N_THREADS = 4                              # descargas concurrentes (cortesía con el servidor)
REQUEST_TIMEOUT = 20
N_RETRIES = 3
SEED = 42

ROAD_WIDTH_MODE = "by_class"               # "by_class" (ancho por clase, en metros) | "fixed" (como Massachusetts)
FIXED_WIDTH_PX = 7                         # ancho fijo en px si ROAD_WIDTH_MODE == "fixed"
DRIVABLE = ["motorway", "trunk", "primary", "secondary", "tertiary",
            "residential", "living_street", "unclassified", "service"]
WIDTH_M = {"motorway": 20, "trunk": 18, "primary": 14, "secondary": 12, "tertiary": 10,
           "residential": 7, "living_street": 6, "unclassified": 6, "service": 4}

SPLIT_RATIOS = (0.70, 0.15, 0.15)          # train / val / test (por bloques espaciales)
BLOCK_SIZE = 4                             # parches por lado de bloque espacial

# ----------------------------- Auditoría de alineación -----------------------------
SHIFT_R = 12                               # radio de búsqueda de desplazamientos (px)
MIN_BOUNDARY_PIX = 200                     # mínimo de píxeles de borde utilizables por parche
CONF_MIN = 2.0                             # confianza mínima del pico para entrar a las estadísticas
MAX_BOUNDARY_PTS = 4000                    # submuestreo de píxeles de borde (velocidad)
N_AUDIT_PATCHES = 300                      # parches Overture a auditar (None = todos)
LIKE_THR = 0.6                             # umbral de "road-likeness" alto (proxy de completitud)
LIKE_DILATE_PX = 6                         # dilatación de la máscara para el proxy de completitud
IOU_MAX_SHIFT = 10                         # desplazamiento máximo (px) de la curva IoU
APPLY_OFFSET_CORRECTION = False            # True: escribe masks_aligned/ con el offset estimado
N_MANUAL_QA = 30                           # parches para la plantilla de QA manual

# ----------------------------- Massachusetts Roads (Parte C) -----------------------------
MASS_M_PER_PX = 1.0                        # resolución aproximada de Massachusetts (~1 m/px)
MASS_BLANK_THR = 0.25                      # se descartan recortes con >25 % de píxeles en blanco
MASS_N_AUDIT_CROPS = 200                   # recortes 512x512 para la auditoría
MASS_N_EDA_TILES = 60                      # tiles para estadísticas de EDA

# ----------------------------- Rutas de salida (todo en /kaggle/working) -----------------------------
WORK = "/kaggle/working"
OUT_DIR = f"{WORK}/overture_lima"
IMG_DIR = f"{OUT_DIR}/images"
MASK_DIR = f"{OUT_DIR}/masks"
MASK_ALIGNED_DIR = f"{OUT_DIR}/masks_aligned"
TILE_CACHE = f"{WORK}/cache/esri_tiles"
SEGMENTS_PATH = f"{WORK}/cache/segments.parquet"
RESULTS_DIR = f"{WORK}/results"
FIG_DIR = f"{WORK}/figures"

In [ ]:
import os, io, sys, json, math, time, glob, shutil, random, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from PIL import Image
from scipy import ndimage as ndi
import geopandas as gpd
from shapely import wkb
from shapely.geometry import box
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from matplotlib.colors import rgb_to_hsv
from tqdm.auto import tqdm


def seed_everything(seed):
    """Fija todas las semillas relevantes para reproducibilidad."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


seed_everything(SEED)
for d in [IMG_DIR, MASK_DIR, TILE_CACHE, RESULTS_DIR, FIG_DIR, os.path.dirname(SEGMENTS_PATH)]:
    os.makedirs(d, exist_ok=True)


def save_fig(name):
    """Guarda la figura actual en FIG_DIR y la muestra."""
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=130, bbox_inches="tight")
    plt.show()


print("rasterio", rasterio.__version__, "| geopandas", gpd.__version__, "| numpy", np.__version__)

# PARTE A · Dataset propio: Overture Maps + Esri World Imagery

## A1. Descarga de segmentos viales de Overture
Overture publica la red vial como *segments* (líneas) en GeoParquet, sin imágenes. Se descarga solo el bbox con el cliente oficial `overturemaps` (consulta directa al bucket público; requiere Internet). El comando exacto se imprime para dejar trazabilidad.

Referencia: <https://docs.overturemaps.org/getting-data/>.

In [ ]:
def download_segments(bbox, out_path):
    """Descarga segmentos de transporte de Overture para un bbox (lon_min, lat_min, lon_max, lat_max)."""
    exe = shutil.which("overturemaps")
    if exe is None:
        raise RuntimeError("No se encontró el comando 'overturemaps'. Ejecuta la celda de instalación "
                           "(pip install overturemaps) y verifica que Internet esté en On.")
    cmd = [exe, "download", f"--bbox={bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}",
           "-f", "geoparquet", "--type=segment", "-o", out_path]
    print("Comando:", " ".join(cmd))
    subprocess.run(cmd, check=True)


if not os.path.exists(SEGMENTS_PATH):
    t0 = time.time()
    download_segments(BBOX, SEGMENTS_PATH)
    print(f"Descarga completada en {time.time() - t0:.0f} s")
else:
    print("Se reutiliza el archivo existente:", SEGMENTS_PATH)
print(f"Tamaño: {os.path.getsize(SEGMENTS_PATH) / 1e6:.1f} MB")

In [ ]:
def load_segments(path):
    """Lee el GeoParquet; si la geometría viene como WKB (bytes), la decodifica con shapely.wkb."""
    try:
        gdf = gpd.read_parquet(path)
    except Exception as exc:  # geometría sin metadata geo
        print("read_parquet geoespacial falló, se decodifica WKB manualmente:", exc)
        df = pd.read_parquet(path)
        geoms = df["geometry"].apply(lambda b: wkb.loads(bytes(b)))
        gdf = gpd.GeoDataFrame(df.drop(columns=["geometry"]), geometry=geoms, crs="EPSG:4326")
    if gdf.geometry.dtype == object and len(gdf) and isinstance(gdf.geometry.iloc[0], (bytes, bytearray)):
        gdf["geometry"] = gdf.geometry.apply(lambda b: wkb.loads(bytes(b)))
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    return gdf


def as_text(value):
    """Normaliza valores posiblemente anidados/ausentes a texto simple."""
    if isinstance(value, str):
        return value
    return "unknown"


def ensure_text_col(gdf, name):
    """Garantiza que la columna exista y sea texto (defensivo: Overture cambia de esquema)."""
    if name not in gdf.columns:
        print(f"Aviso: la columna '{name}' no existe en esta release de Overture; se usa 'unknown'.")
        gdf[name] = "unknown"
    gdf[name] = gdf[name].apply(as_text)


def surface_of(value):
    """road_surface suele ser una lista de dicts [{'value': 'paved', ...}]; extrae el primer valor."""
    try:
        if value is None:
            return "unknown"
        if isinstance(value, str):
            return value
        first = list(value)[0]
        return first.get("value", "unknown") if isinstance(first, dict) else "unknown"
    except Exception:
        return "unknown"


gdf = load_segments(SEGMENTS_PATH)
for col in ["subtype", "class"]:
    ensure_text_col(gdf, col)
print("Segmentos:", len(gdf), "| CRS:", gdf.crs)
print("Columnas:", list(gdf.columns))
gdf.head(3)

## A2. EDA vectorial
**Jerarquía de Overture (tema `transportation`, tipo `segment`):** `subtype` indica el modo (`road`, `rail`, `water`); dentro de `road`, `class` describe la función de la vía, de mayor a menor jerarquía: `motorway`, `trunk`, `primary`, `secondary`, `tertiary`, `residential`, `living_street`, `unclassified`, `service`, y las vías no vehiculares `pedestrian`, `footway`, `steps`, `path`, `cycleway`, `bridleway`, `track`, además de `unknown`. La longitud se calcula en un CRS métrico (UTM estimado), no en grados.

In [ ]:
utm = gdf.estimate_utm_crs()
gdf["length_km"] = gdf.to_crs(utm).length / 1000.0
print("CRS métrico usado para longitudes:", utm.to_string())

by_sub = gdf.groupby("subtype").agg(n=("subtype", "size"), km=("length_km", "sum")).sort_values("n", ascending=False)
print("\nSegmentos por subtype:")
display(by_sub.round(1))

roads_all = gdf[gdf["subtype"] == "road"] if (gdf["subtype"] == "road").any() else gdf
by_cls = (roads_all.groupby("class").agg(n=("class", "size"), km=("length_km", "sum"))
          .sort_values("km", ascending=False))
by_cls["pct_km"] = 100 * by_cls["km"] / by_cls["km"].sum()
print("\nSegmentos y longitud por class (subtype=road):")
display(by_cls.round(1))

if "road_surface" in roads_all.columns:
    surf = roads_all["road_surface"].apply(surface_of).value_counts()
    print("\nSuperficie declarada (road_surface):")
    display(surf.head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
by_cls["n"].plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_title("Segmentos por class"); axes[0].set_ylabel("N.º de segmentos")
by_cls["km"].plot.bar(ax=axes[1], color="#55A868")
axes[1].set_title("Longitud total por class"); axes[1].set_ylabel("km")
save_fig("a2_class_counts")

fig, ax = plt.subplots(figsize=(8, 8))
roads_all.plot(column="class", ax=ax, legend=True, linewidth=0.5, cmap="tab20",
               legend_kwds={"loc": "upper left", "fontsize": 7})
ax.set_title("Segmentos viales Overture por class"); ax.set_xlabel("lon"); ax.set_ylabel("lat")
save_fig("a2_class_map")

**Cómo leer este resultado:** en ciudades como Lima domina `residential` en conteo, pero las vías `primary`/`secondary` aportan mucha longitud por segmento y, al ser más anchas, más píxeles de máscara. Una proporción alta de `service`/`unclassified`/`unknown` anticipa etiquetas más inciertas (callejones, accesos, vías informales).

## A3. Filtro de clases conducibles y anchos por clase
Se conservan `subtype == "road"` con clases vehiculares y se **excluyen** `footway`, `path`, `steps`, `cycleway`, `bridleway`, `track`, `pedestrian` y `unknown`. Justificación: a ~1 m/px un sendero peatonal o una trocha mide 1–2 px, queda tapado por árboles/edificios o es ambiguo para el ojo humano; incluirlos inyectaría etiquetas que ni siquiera un anotador podría verificar. Los anchos en metros (tabla en CONFIG) son valores representativos por clase, no mediciones; el modo `fixed` replica el ancho constante de Massachusetts para comparar.

In [ ]:
mask_drive = (gdf["subtype"] == "road") if (gdf["subtype"] == "road").any() else pd.Series(True, index=gdf.index)
roads = gdf[mask_drive & gdf["class"].isin(DRIVABLE)].copy()
roads["width_m"] = roads["class"].map(WIDTH_M).astype(float)
print(f"Segmentos conducibles: {len(roads)} de {len(gdf)} ({100 * len(roads) / len(gdf):.1f} %)")
print(f"Longitud conducible: {roads['length_km'].sum():.0f} km")
if len(roads) == 0:
    raise RuntimeError("No quedaron segmentos tras el filtro; revisa BBOX o los nombres de class en esta release.")

roads3857 = roads.to_crs(3857).reset_index(drop=True)
sindex = roads3857.sindex          # índice espacial (R-tree) para consultar segmentos por parche
print("Índice espacial listo:", len(roads3857), "geometrías en EPSG:3857")

## A4. Matemática de tiles (implementación manual, sin `mercantile`)
El esquema XYZ (Web Mercator, EPSG:3857) divide el mundo en `2^z × 2^z` tiles de 256 px. Con `ORIGIN = 20037508.34 m`, el lado de un tile es `2·ORIGIN / 2^z` en unidades de EPSG:3857. Esas unidades **no son metros reales** salvo en el ecuador: la escala real es `cos(lat)`, por lo que la resolución en suelo es `≈ 156543.03·cos(lat) / 2^z` m/px. Un parche es un bloque de 2×2 tiles alineado a índices pares, de modo que (px, py) ↔ tiles (x..x+1, y..y+1).

In [ ]:
ORIGIN = 20037508.342789244
EARTH_R = 6378137.0


def lonlat_to_tile(lon, lat, z):
    """lon/lat (grados) -> índices de tile XYZ (x, y) en el zoom z."""
    n = 2 ** z
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2.0 * n)
    return x, y


def tile_bounds_3857(x, y, z):
    """Límites (minx, miny, maxx, maxy) de un tile en EPSG:3857."""
    size = 2 * ORIGIN / 2 ** z
    minx = -ORIGIN + x * size
    maxy = ORIGIN - y * size
    return minx, maxy - size, minx + size, maxy


def patch_bounds_3857(x, y, z):
    """Límites del parche formado por los tiles (x..x+1, y..y+1)."""
    minx, _, _, maxy = tile_bounds_3857(x, y, z)
    _, miny, maxx, _ = tile_bounds_3857(x + 1, y + 1, z)
    return minx, miny, maxx, maxy


def merc_to_lonlat(mx, my):
    lon = math.degrees(mx / EARTH_R)
    lat = math.degrees(2 * math.atan(math.exp(my / EARTH_R)) - math.pi / 2)
    return lon, lat


def ground_resolution(lat, z):
    """Metros reales por píxel (tile de 256 px) a una latitud dada."""
    return 156543.03392804097 * math.cos(math.radians(lat)) / 2 ** z


lat_c = (BBOX[1] + BBOX[3]) / 2
print(f"Resolución en suelo a lat {lat_c:.3f}°, z={ZOOM}: {ground_resolution(lat_c, ZOOM):.3f} m/px")
print(f"Lado de un parche (512 px) ≈ {PATCH * ground_resolution(lat_c, ZOOM):.0f} m")


def build_patch_list(bbox, zoom, max_patches, seed):
    """Lista de parches (x, y) alineados a índices pares que cubren el bbox (submuestreo reproducible)."""
    x_min, y_max = lonlat_to_tile(bbox[0], bbox[1], zoom)   # esquina SO: y crece hacia el sur
    x_max, y_min = lonlat_to_tile(bbox[2], bbox[3], zoom)   # esquina NE
    x0, y0 = x_min - x_min % 2, y_min - y_min % 2
    patches = [(x, y) for x in range(x0, x_max + 1, 2) for y in range(y0, y_max + 1, 2)]
    total = len(patches)
    if max_patches and total > max_patches:
        rng = np.random.default_rng(seed)
        keep = np.sort(rng.choice(total, size=max_patches, replace=False))
        patches = [patches[i] for i in keep]
    return patches, (x0, y0), total


patch_list, GRID_ORIGIN, n_total = build_patch_list(BBOX, ZOOM, MAX_PATCHES, SEED)
print(f"Parches en el bbox: {n_total} | seleccionados: {len(patch_list)} | tiles a descargar ≤ {4 * len(patch_list)}")

## A5. Descarga de imágenes Esri y composición de parches
Se descargan los tiles con `ThreadPoolExecutor` (pocas hebras, con `User-Agent`, *timeout*, 3 reintentos con *backoff* exponencial y caché en disco). Los tiles que fallan se omiten y se reporta su número; un parche se descarta si le falta alguno de sus 4 tiles.

> **Atribución:** *Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community.*
>
> **Aviso para el equipo:** los términos de uso de Esri World Imagery para entrenamiento de modelos / uso académico **deben ser verificados por el equipo** antes de redistribuir las imágenes o publicar resultados derivados. Este notebook no asume que esté permitido; no se debe publicar el dataset de imágenes sin esa verificación.

In [ ]:
HEADERS = {"User-Agent": "dl-cnn-vs-dinov2-academic/1.0 (university course project)"}


def fetch_tile(z, x, y, session):
    """Descarga un tile con caché, reintentos y backoff. Devuelve la ruta o None si falla."""
    path = f"{TILE_CACHE}/{z}_{x}_{y}.jpg"
    if os.path.exists(path) and os.path.getsize(path) > 500:
        return path
    url = ESRI_URL.format(z=z, x=x, y=y)
    for attempt in range(N_RETRIES):
        try:
            r = session.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            ctype = r.headers.get("Content-Type", "")
            if r.status_code == 200 and "image" in ctype and len(r.content) > 500:
                with open(path, "wb") as f:
                    f.write(r.content)
                return path
        except requests.RequestException:
            pass
        time.sleep(2 ** attempt)
    return None


def download_tiles(patches, z):
    """Descarga todos los tiles necesarios en paralelo. Devuelve {(x, y): ruta o None}."""
    needed = sorted({(x + dx, y + dy) for x, y in patches for dx in (0, 1) for dy in (0, 1)})
    results = {}
    with requests.Session() as session, ThreadPoolExecutor(max_workers=N_THREADS) as pool:
        futures = {pool.submit(fetch_tile, z, x, y, session): (x, y) for x, y in needed}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Tiles Esri"):
            results[futures[fut]] = fut.result()
    failed = sum(v is None for v in results.values())
    print(f"Tiles solicitados: {len(needed)} | fallidos: {failed}")
    return results


def stitch_patch(x, y, tiles):
    """Une 2x2 tiles en una imagen RGB de 512x512. Devuelve None si falta algún tile."""
    canvas = Image.new("RGB", (PATCH, PATCH))
    for dx in (0, 1):
        for dy in (0, 1):
            p = tiles.get((x + dx, y + dy))
            if p is None:
                return None
            canvas.paste(Image.open(p).convert("RGB"), (dx * TILE, dy * TILE))
    return canvas


tiles = download_tiles(patch_list, ZOOM)

## A6. Rasterización de máscaras
Para cada parche se consultan los segmentos que lo intersectan (índice espacial), se *bufferizan* y se rasterizan con `rasterio.features.rasterize` usando `from_bounds` (transformación afín exacta entre EPSG:3857 y los 512×512 píxeles).

**Por qué se divide por `cos(lat)`:** las unidades de EPSG:3857 se estiran en la dirección horizontal y vertical por un factor `1/cos(lat)` respecto del suelo real. A −12° eso es ≈ 2.2 %; un ancho de 7 m en el suelo equivale a `7/cos(lat)` unidades de 3857. Sin esta corrección las vías saldrían ligeramente más delgadas que su ancho real y el efecto crecería con la latitud. En modo `fixed` el ancho se define directamente en píxeles (`FIXED_WIDTH_PX · tamaño_de_píxel_3857`).

In [ ]:
def rasterize_patch(bounds):
    """Máscara binaria uint8 {0,1} de 512x512 para un parche dado por sus límites EPSG:3857."""
    minx, miny, maxx, maxy = bounds
    lat = merc_to_lonlat(0.0, (miny + maxy) / 2)[1]
    cos_lat = math.cos(math.radians(lat))
    px3857 = (maxx - minx) / PATCH                     # tamaño de píxel en unidades 3857
    margin = 15.0 / cos_lat                            # holgura >= medio ancho máximo (10 m)
    idx = sindex.query(box(minx - margin, miny - margin, maxx + margin, maxy + margin), predicate="intersects")
    n_seg = int(len(idx))
    if n_seg == 0:
        return np.zeros((PATCH, PATCH), dtype=np.uint8), 0
    sub = roads3857.iloc[idx]
    if ROAD_WIDTH_MODE == "fixed":
        half = np.full(len(sub), FIXED_WIDTH_PX * px3857 / 2.0)
    else:
        half = sub["width_m"].to_numpy() / 2.0 / cos_lat
    shapes = [(g.buffer(h), 1) for g, h in zip(sub.geometry, half)]
    transform = from_bounds(minx, miny, maxx, maxy, PATCH, PATCH)
    mask = rasterize(shapes, out_shape=(PATCH, PATCH), transform=transform, fill=0, dtype="uint8")
    return mask, n_seg


def build_dataset(patches, tiles):
    """Genera imagen + máscara por parche, los guarda como PNG y devuelve el DataFrame de metadatos."""
    rows = []
    for x, y in tqdm(patches, desc="Parches"):
        img = stitch_patch(x, y, tiles)
        if img is None:
            continue
        bounds = patch_bounds_3857(x, y, ZOOM)
        mask, n_seg = rasterize_patch(bounds)
        pid = f"z{ZOOM}_{x}_{y}"
        img.save(f"{IMG_DIR}/{pid}.png")
        Image.fromarray(mask * 255).save(f"{MASK_DIR}/{pid}.png")
        cx, cy = (bounds[0] + bounds[2]) / 2, (bounds[1] + bounds[3]) / 2
        lon, lat = merc_to_lonlat(cx, cy)
        rows.append({"id": pid, "tile_x": x, "tile_y": y, "lon": lon, "lat": lat,
                     "b3857_minx": bounds[0], "b3857_miny": bounds[1],
                     "b3857_maxx": bounds[2], "b3857_maxy": bounds[3],
                     "m_per_px": ground_resolution(lat, ZOOM),
                     "road_ratio": float(mask.mean()), "n_segments": n_seg})
    return pd.DataFrame(rows)


meta = build_dataset(patch_list, tiles)
print(f"Parches generados: {len(meta)} de {len(patch_list)} solicitados")
meta.head()

## A7. Split espacial por bloques (sin fuga de información)
Si se reparten parches al azar, parches vecinos (que comparten la misma calle, el mismo tipo de techo y el mismo sensor/fecha) caen en train y test: el modelo "reconoce" la zona y el test sobreestima el rendimiento (*data leakage* espacial). Se agrupan los parches en bloques de `BLOCK_SIZE × BLOCK_SIZE` parches, se barajan los **bloques** con `SEED` y se asignan completos a train/val/test según `SPLIT_RATIOS`.

In [ ]:
def assign_spatial_split(df, grid_origin, block_size, ratios, seed):
    """Asigna split por bloque espacial. Devuelve una Serie con 'train'/'val'/'test'."""
    px = (df["tile_x"] - grid_origin[0]) // 2
    py = (df["tile_y"] - grid_origin[1]) // 2
    block = list(zip((px // block_size).tolist(), (py // block_size).tolist()))
    unique_blocks = sorted(set(block))
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(unique_blocks))
    n = len(unique_blocks)
    n_val = max(1, round(ratios[1] * n))
    n_test = max(1, round(ratios[2] * n))
    split_of = {}
    for rank, bi in enumerate(order):
        split_of[unique_blocks[bi]] = "test" if rank < n_test else ("val" if rank < n_test + n_val else "train")
    return pd.Series([split_of[b] for b in block], index=df.index), len(unique_blocks)


meta["split"], n_blocks = assign_spatial_split(meta, GRID_ORIGIN, BLOCK_SIZE, SPLIT_RATIOS, SEED)
print(f"Bloques espaciales: {n_blocks}")
print(meta["split"].value_counts().rename("n_parches").to_frame().assign(pct=lambda d: (100 * d.n_parches / len(meta)).round(1)))

colors = {"train": "#4C72B0", "val": "#DD8452", "test": "#55A868"}
fig, ax = plt.subplots(figsize=(7, 7))
for s, g in meta.groupby("split"):
    ax.scatter(g["lon"], g["lat"], s=22, c=colors[s], label=f"{s} (n={len(g)})", marker="s")
ax.set_title("Split espacial por bloques"); ax.set_xlabel("lon"); ax.set_ylabel("lat"); ax.legend()
ax.set_aspect(1 / math.cos(math.radians(lat_c)))
save_fig("a7_split_map")

meta.to_csv(f"{OUT_DIR}/patches.csv", index=False)
print("Escrito:", f"{OUT_DIR}/patches.csv")

**Cómo leer este resultado:** los cuadrados del mismo color deben formar manchas contiguas (bloques); si ves colores mezclados parche a parche, el split no está protegiendo contra fuga. Con `MAX_PATCHES` bajo los bloques quedan ralos, lo cual es esperado.

## A8. EDA de parches
La fracción de píxeles de vía es la medida directa del **desbalance de clases**: si es pequeña, una pérdida BCE sola favorece predecir "todo fondo", por eso el proyecto usa BCE + Dice.

In [ ]:
def load_img(pid, folder=IMG_DIR):
    return np.array(Image.open(f"{folder}/{pid}.png").convert("RGB"))


def load_mask(pid, folder=MASK_DIR):
    return (np.array(Image.open(f"{folder}/{pid}.png").convert("L")) > 127).astype(np.uint8)


def overlay_mask(ax, img, mask, color=(1.0, 0.0, 0.0), alpha=0.45):
    """Dibuja la máscara semitransparente en rojo sobre la imagen."""
    ov = img.astype(np.float32) / 255.0
    m = mask.astype(bool)
    ov[m] = (1 - alpha) * ov[m] + alpha * np.array(color)
    ax.imshow(ov); ax.axis("off")


zero_pct = 100 * (meta["road_ratio"] == 0).mean()
print(f"Fracción media de píxeles de vía: {100 * meta['road_ratio'].mean():.2f} %")
print(f"Mediana: {100 * meta['road_ratio'].median():.2f} % | parches sin vía: {zero_pct:.1f} %")
print(f"Desbalance aproximado fondo:vía = {(1 - meta['road_ratio'].mean()) / max(meta['road_ratio'].mean(), 1e-9):.1f} : 1")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(100 * meta["road_ratio"], bins=40, color="#4C72B0")
ax.set_xlabel("% de píxeles de vía por parche"); ax.set_ylabel("N.º de parches"); ax.set_title("Distribución de la fracción de vía")
save_fig("a8_road_ratio_hist")

In [ ]:
def show_grid(ids, title, fname, ncols=4):
    nrows = math.ceil(len(ids) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, pid in zip(np.atleast_1d(axes).ravel(), ids):
        overlay_mask(ax, load_img(pid), load_mask(pid))
        r = meta.loc[meta.id == pid, "road_ratio"].iloc[0]
        ax.set_title(f"{pid}\n{100 * r:.1f} % vía", fontsize=7)
    fig.suptitle(title, y=1.0)
    save_fig(fname)


rng = np.random.default_rng(SEED)
show_grid(rng.choice(meta["id"], size=min(8, len(meta)), replace=False), "Ejemplos aleatorios (máscara en rojo)", "a8_examples")
ordered = meta.sort_values("road_ratio")
show_grid(list(ordered["id"].tail(4)) + list(ordered["id"].head(4)),
          "Mayor fracción de vía (arriba) y menor (abajo)", "a8_extremes")

**Cómo leer este resultado:** las vías deben quedar cubiertas por el rojo. Si el rojo aparece sistemáticamente corrido respecto de las calles, es un problema de registro: la Parte B lo cuantifica. Parches de mayor fracción suelen ser centros densos; los de menor, zonas con agua, cerros o terrenos baldíos.

# PARTE B · Auditoría de alineación (Overture vs Esri)

Hay varias fuentes de desalineación entre un vector y una imagen: error de posición del dato original (digitalizado sobre otro mosaico o fecha), distorsión por ortorrectificación del mosaico Esri (edificios y árboles altos desplazados respecto del suelo), y diferencias de datum o de generalización. Todas se manifiestan como un desplazamiento local entre el borde de la máscara y el borde real de la calle.

**Convención de signos:** `(dx, dy)` es el desplazamiento que hay que **aplicar a la máscara** para que coincida con la imagen. `dx > 0` mueve la máscara al este (derecha) y `dy > 0` al sur (hacia abajo, filas crecientes). 1 px ≈ 1.17 m en Lima a z17.

## Qué mide cada métrica

| # | Métrica | Qué mide | Cómo interpretarla |
|---|---|---|---|
| 1 | **QA visual** (contornos) | Inspección humana de si el borde de la máscara cae sobre el borde de la calzada. | Referencia cualitativa; detecta patrones que los números no (p. ej. calles ausentes). |
| 2 | **Búsqueda de desplazamiento por bordes** (principal) | Se calcula la magnitud del gradiente de la imagen (Sobel sobre gris suavizado) y se promedia sobre el borde de la máscara desplazado en una ventana ±R px. El desplazamiento que maximiza el promedio es el que mejor alinea borde de máscara con bordes de la imagen. | **Sesgo sistemático** = mediana de (dx, dy): error corregible por traslación global. **Dispersión** = MAD/desv. estándar: error aleatorio, no corregible. Heurística: mediana \|offset\| ≤ 2 px (≈ 2.3 m) es buena para calles de 7 m; > 4 px compromete el IoU (ver 4). |
| 2b | **Confianza del pico** | `(S_max − mediana(S)) / (std(S) + ε)`: cuán destacado es el mejor desplazamiento frente al resto de la ventana. | Pico plano = poca información (zona sin bordes claros); solo se usan parches con confianza ≥ `CONF_MIN`. **Ganancia** = `S(mejor)/S(0,0)`: cuánto mejora el ajuste al corregir. |
| 3 | **Correlación cruzada de "road-likeness"** | Proxy independiente: mapa no supervisado de píxeles grises de brillo medio (calzada) y correlación normalizada con la máscara desplazada. | La **tasa de acuerdo** entre dos proxies independientes (bordes vs color) sube la confianza: si ambos coinciden en el offset, es poco probable que sea un artefacto. El proxy es tosco (veredas, techos y suelo gris también puntúan). |
| 4 | **Curva IoU vs desplazamiento** | IoU entre la máscara y ella misma desplazada `d` px (promedio de 4 direcciones). | Es un **techo**: con un error de registro de `d` px, ni un modelo perfecto supera ese IoU. Motiva la métrica relajada (tolerancia ρ) del notebook 02. |
| 5 | **Proxy de completitud** | Fracción de píxeles de alta "road-likeness" fuera de la máscara dilatada. | Indicador grueso de calles faltantes (informales, sin pavimentar); no es una medida de error, solo una alerta comparativa. |
| 6 | **Corrección opcional** | Aplica el offset estimado y escribe `masks_aligned/`. | Riesgo: sobreajustar al proxy y "mover" etiquetas correctas. Úsese solo con evidencia de sesgo sistemático claro. |
| 7 | **QA manual** | Plantilla CSV + hoja de contacto de 30 parches para que el equipo etiquete. | Es la única validación independiente de los proxies automáticos. |

In [ ]:
EPS = 1e-6


def gradient_magnitude(rgb, sigma=1.5):
    """Magnitud de gradiente Sobel sobre gris suavizado, normalizada por parche (percentil 99 -> 1)."""
    gray = (rgb.astype(np.float32) @ np.array([0.299, 0.587, 0.114], dtype=np.float32)) / 255.0
    g = ndi.gaussian_filter(gray, sigma)
    G = np.hypot(ndi.sobel(g, axis=1), ndi.sobel(g, axis=0))
    return np.clip(G / (np.percentile(G, 99) + EPS), 0, 1).astype(np.float32)


def mask_boundary(mask):
    """Borde de la máscara: B = M XOR erosión(M)."""
    m = mask.astype(bool)
    return m & ~ndi.binary_erosion(m)


def edge_score_map(G, boundary, R, max_pts=MAX_BOUNDARY_PTS, min_pts=MIN_BOUNDARY_PIX):
    """S(dx, dy) = media de G sobre los píxeles del borde desplazados (dx, dy), con |dx|,|dy| <= R.

    Devuelve (S, n_borde). S[i, j] corresponde a dy = i - R, dx = j - R. Se descartan píxeles de borde
    a menos de R px del límite del parche para que ningún desplazamiento salga de la imagen.
    """
    H, W = G.shape
    ys, xs = np.nonzero(boundary)
    keep = (ys >= R) & (ys < H - R) & (xs >= R) & (xs < W - R)
    ys, xs = ys[keep], xs[keep]
    n_b = int(len(ys))
    if n_b < min_pts:
        return None, n_b
    if n_b > max_pts:
        sel = np.random.default_rng(0).choice(n_b, size=max_pts, replace=False)
        ys, xs = ys[sel], xs[sel]
    S = np.empty((2 * R + 1, 2 * R + 1), dtype=np.float32)
    for i, dy in enumerate(range(-R, R + 1)):
        for j, dx in enumerate(range(-R, R + 1)):
            S[i, j] = G[ys + dy, xs + dx].mean()
    return S, n_b


def summarize_score_map(S, R):
    """Mejor desplazamiento, ganancia y confianza de un mapa de puntajes."""
    i, j = np.unravel_index(int(np.argmax(S)), S.shape)
    dy, dx = int(i - R), int(j - R)
    s_max, s0 = float(S[i, j]), float(S[R, R])
    conf = (s_max - float(np.median(S))) / (float(S.std()) + EPS)
    return {"dx": dx, "dy": dy, "gain": s_max / (s0 + EPS), "conf": conf,
            "at_window_edge": bool(max(abs(dx), abs(dy)) == R)}


def road_likeness(rgb, sigma=2.0):
    """Mapa no supervisado de 'parece calzada': baja saturación y brillo medio (gris), suavizado."""
    hsv = rgb_to_hsv(rgb.astype(np.float32) / 255.0)
    sat, val = hsv[..., 1], hsv[..., 2]
    like = (1.0 - sat) * np.exp(-((val - 0.45) / 0.25) ** 2)
    return ndi.gaussian_filter(like, sigma).astype(np.float32)


def ncc_score_map(like, mask, R, stride=2):
    """Correlación normalizada (Pearson) entre 'like' y la máscara desplazada (dx, dy), |.| <= R."""
    H, W = mask.shape
    a = like[R:H - R:stride, R:W - R:stride].astype(np.float32)
    a = a - a.mean()
    a_norm = float(np.sqrt((a * a).sum())) + EPS
    m = mask.astype(np.float32)
    S = np.zeros((2 * R + 1, 2 * R + 1), dtype=np.float32)
    for i, dy in enumerate(range(-R, R + 1)):
        for j, dx in enumerate(range(-R, R + 1)):
            b = m[R - dy:H - R - dy:stride, R - dx:W - R - dx:stride]   # máscara desplazada (dx, dy)
            b = b - b.mean()
            S[i, j] = float((a * b).sum()) / (a_norm * (float(np.sqrt((b * b).sum())) + EPS))
    return S


def missing_road_fraction(like, mask, thr=LIKE_THR, dilate_px=LIKE_DILATE_PX):
    """Fracción de píxeles de alta road-likeness fuera de la máscara dilatada (proxy de calles faltantes)."""
    high = like > thr
    near = ndi.binary_dilation(mask > 0, iterations=dilate_px)
    n_high = int(high.sum())
    return (float((high & ~near).sum()) / n_high) if n_high else float("nan")


def audit_patch(pid, rgb, mask, R=SHIFT_R):
    """Auditoría de un parche: offset por bordes, offset por NCC, confianza, ganancia y completitud."""
    nan = float("nan")
    row = {"id": pid, "valid": False, "n_boundary": 0, "dx": nan, "dy": nan, "gain": nan, "conf": nan,
           "at_window_edge": False, "ncc_dx": nan, "ncc_dy": nan, "ncc_conf": nan, "missing_frac": nan}
    S, n_b = edge_score_map(gradient_magnitude(rgb), mask_boundary(mask), R)
    row["n_boundary"] = n_b
    if S is None:
        return row
    row.update(summarize_score_map(S, R))
    row["valid"] = True
    like = road_likeness(rgb)
    N = ncc_score_map(like, mask, R)
    nb = summarize_score_map(N, R)
    row.update({"ncc_dx": nb["dx"], "ncc_dy": nb["dy"], "ncc_conf": nb["conf"]})
    row["missing_frac"] = missing_road_fraction(like, mask)
    return row


def run_audit(items, total=None):
    """Aplica audit_patch a un iterable de (id, rgb, mask01)."""
    return pd.DataFrame([audit_patch(pid, rgb, m) for pid, rgb, m in tqdm(items, total=total, desc="Auditoría")])


def shift_mask(m, dx, dy):
    """Desplaza la máscara (dx, dy) px rellenando con ceros (sin envolver)."""
    H, W = m.shape
    out = np.zeros_like(m)
    sy, dy_ = slice(max(0, -dy), min(H, H - dy)), slice(max(0, dy), min(H, H + dy))
    sx, dx_ = slice(max(0, -dx), min(W, W - dx)), slice(max(0, dx), min(W, W + dx))
    out[dy_, dx_] = m[sy, sx]
    return out


def iou_bool(a, b):
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else float("nan")


def iou_sensitivity(masks, dmax=IOU_MAX_SHIFT):
    """IoU(mask, desplazada d px) promediado sobre parches y las 4 direcciones de los ejes."""
    curve = []
    for d in range(dmax + 1):
        vals = [iou_bool(m > 0, shift_mask(m, dx, dy) > 0)
                for m in masks for dx, dy in [(d, 0), (-d, 0), (0, d), (0, -d)]]
        curve.append(float(np.nanmean(vals)))
    return curve


def mad_std(v):
    v = np.asarray(v, dtype=float)
    return float(1.4826 * np.median(np.abs(v - np.median(v)))) if len(v) else float("nan")


def summarize_audit(df, label):
    """Estadísticas robustas sobre los parches con pico confiable."""
    ok = df[df["valid"] & (df["conf"] >= CONF_MIN)].copy()
    out = {"label": label, "n_patches": int(len(df)), "n_valid_boundary": int(df["valid"].sum()),
           "n_confident": int(len(ok)), "mean_road_ratio": float(df.get("road_ratio", pd.Series([np.nan])).mean())}
    if len(ok) == 0:
        return out
    off_px = np.hypot(ok["dx"], ok["dy"])
    off_m = off_px * ok["m_per_px"]
    d_agree = np.hypot(ok["dx"] - ok["ncc_dx"], ok["dy"] - ok["ncc_dy"])
    out.update({
        "median_dx_px": float(ok["dx"].median()), "median_dy_px": float(ok["dy"].median()),
        "robust_std_dx_px": mad_std(ok["dx"]), "robust_std_dy_px": mad_std(ok["dy"]),
        "median_abs_offset_px": float(off_px.median()), "median_abs_offset_m": float(off_m.median()),
        "p90_abs_offset_px": float(np.percentile(off_px, 90)),
        "pct_le_2px": float(100 * (off_px <= 2).mean()), "pct_le_4px": float(100 * (off_px <= 4).mean()),
        "pct_gt_6px": float(100 * (off_px > 6).mean()),
        "median_gain": float(ok["gain"].median()),
        "pct_peak_at_window_edge": float(100 * ok["at_window_edge"].mean()),
        "ncc_agreement_pct_le_3px": float(100 * (d_agree <= 3).mean()),
        "median_missing_frac": float(ok["missing_frac"].median()),
    })
    return out


def draw_contour(ax, img, mask, color="red", lw=0.7):
    """Imagen con el CONTORNO de la máscara (no relleno), para ver si el borde cae sobre la calzada."""
    ax.imshow(img)
    if mask.any():
        ax.contour(mask.astype(float), levels=[0.5], colors=color, linewidths=lw)
    ax.axis("off")


print("Funciones de auditoría definidas (R =", SHIFT_R, "px).")

## B1. QA visual: contornos de la máscara sobre la imagen
Se dibuja solo el **contorno** de la máscara para no tapar la calzada. Debajo, recortes ampliados alrededor del borde de la máscara para ver si el contorno cae sobre el borde de la calle, dentro de ella o fuera.

In [ ]:
rng = np.random.default_rng(SEED + 1)
cand = meta[meta["road_ratio"] > 0.02]["id"].to_numpy()
qa_ids = rng.choice(cand, size=min(12, len(cand)), replace=False)

fig, axes = plt.subplots(3, 4, figsize=(14, 10.5))
for ax, pid in zip(axes.ravel(), qa_ids):
    draw_contour(ax, load_img(pid), load_mask(pid))
    ax.set_title(pid, fontsize=7)
fig.suptitle("QA visual: contorno de la máscara Overture sobre Esri", y=1.0)
save_fig("b1_contours")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, pid in zip(axes.ravel(), qa_ids[:8]):
    img, m = load_img(pid), load_mask(pid)
    ys, xs = np.nonzero(mask_boundary(m))
    cy, cx = (int(np.median(ys)), int(np.median(xs))) if len(ys) else (PATCH // 2, PATCH // 2)
    y0, x0 = int(np.clip(cy - 64, 0, PATCH - 128)), int(np.clip(cx - 64, 0, PATCH - 128))
    draw_contour(ax, img[y0:y0 + 128, x0:x0 + 128], m[y0:y0 + 128, x0:x0 + 128], lw=1.2)
    ax.set_title(f"{pid} (zoom 128 px)", fontsize=7)
save_fig("b1_zoom")

**Cómo leer este resultado:** un buen alineamiento muestra el contorno rojo pegado al borde claro/oscuro de la calzada, a ambos lados. Un contorno consistentemente corrido hacia un lado indica sesgo sistemático; contornos correctos en unas zonas y corridos en otras indican error aleatorio. Calles visibles sin contorno son vías faltantes en Overture.

## B2. Búsqueda de desplazamiento por bordes (métrica principal)
Se ejecuta la auditoría sobre `N_AUDIT_PATCHES` parches con vía. Cada fila de la tabla resultante tiene el mejor (dx, dy), la ganancia, la confianza y el desplazamiento por correlación de road-likeness.

In [ ]:
audit_pool = meta[meta["road_ratio"] > 0.005]
n_aud = len(audit_pool) if N_AUDIT_PATCHES is None else min(N_AUDIT_PATCHES, len(audit_pool))
audit_ids = audit_pool.sample(n=n_aud, random_state=SEED)["id"].tolist()

items = ((pid, load_img(pid), load_mask(pid)) for pid in audit_ids)
audit_df = run_audit(items, total=len(audit_ids))
audit_df = audit_df.merge(meta[["id", "m_per_px", "road_ratio", "lon", "lat"]], on="id")
audit_df.to_csv(f"{RESULTS_DIR}/alignment_overture_per_patch.csv", index=False)

summary_ov = summarize_audit(audit_df, "overture_lima")
display(pd.Series(summary_ov).to_frame("valor"))

In [ ]:
ok = audit_df[audit_df["valid"] & (audit_df["conf"] >= CONF_MIN)].copy()
ok["off_px"] = np.hypot(ok["dx"], ok["dy"])
ok["off_m"] = ok["off_px"] * ok["m_per_px"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
hb = axes[0].hexbin(ok["dx"], ok["dy"], gridsize=2 * SHIFT_R + 1, extent=(-SHIFT_R, SHIFT_R, -SHIFT_R, SHIFT_R), cmap="viridis", mincnt=1)
plt.colorbar(hb, ax=axes[0], label="n.º de parches")
axes[0].axhline(0, c="w", lw=0.6); axes[0].axvline(0, c="w", lw=0.6)
axes[0].plot(ok["dx"].median(), ok["dy"].median(), "r*", ms=14, label="mediana (sesgo)")
axes[0].invert_yaxis(); axes[0].set_xlabel("dx (px, + = este)"); axes[0].set_ylabel("dy (px, + = sur)")
axes[0].set_title("Offsets por parche"); axes[0].legend(loc="upper right", fontsize=8)

axes[1].hist(ok["off_px"], bins=np.arange(0, SHIFT_R * 1.5, 1), color="#4C72B0")
for t, c in [(2, "g"), (4, "orange"), (6, "r")]:
    axes[1].axvline(t, color=c, ls="--", lw=1)
axes[1].set_xlabel("|offset| (px)"); axes[1].set_ylabel("n.º de parches"); axes[1].set_title("Magnitud del offset (px)")

axes[2].hist(ok["off_m"], bins=30, color="#55A868")
axes[2].set_xlabel("|offset| (m)"); axes[2].set_title("Magnitud del offset (m)")
save_fig("b2_offsets")

print(f"Parches confiables: {len(ok)} de {len(audit_df)}")
print(f"Sesgo sistemático (mediana): dx = {ok['dx'].median():+.1f} px, dy = {ok['dy'].median():+.1f} px "
      f"(= {ok['dx'].median() * ok['m_per_px'].mean():+.1f} m, {ok['dy'].median() * ok['m_per_px'].mean():+.1f} m)")
print(f"Dispersión (desv. robusta MAD): dx = {mad_std(ok['dx']):.1f} px, dy = {mad_std(ok['dy']):.1f} px")
print(f"|offset| <= 2 px: {100 * (ok.off_px <= 2).mean():.1f} % | <= 4 px: {100 * (ok.off_px <= 4).mean():.1f} % | > 6 px: {100 * (ok.off_px > 6).mean():.1f} %")

**Cómo leer este resultado:** el *hexbin* muestra dónde se concentran los offsets. Una nube compacta **fuera** de (0, 0) significa sesgo sistemático (corregible); una nube **ancha** centrada en (0, 0) significa error aleatorio (no corregible). Si muchos picos están en el borde de la ventana (`pct_peak_at_window_edge` alto), el offset real puede superar `SHIFT_R` o el parche no tiene información suficiente. Solo se cuentan parches con confianza ≥ `CONF_MIN`; los parches ambiguos (pocas calles, sombras, mucha vegetación) se descartan para no contaminar la estadística.

### Mapas de puntaje S(dx, dy) en tres ejemplos
Un buen mapa tiene un pico único y estrecho (alta confianza); un mapa plano o con varios picos es ambiguo.

In [ ]:
ranked = ok.sort_values("conf").reset_index(drop=True)
examples = [ranked.iloc[-1], ranked.iloc[len(ranked) // 2], ranked.iloc[0]] if len(ranked) >= 3 else []
fig, axes = plt.subplots(1, len(examples), figsize=(5.5 * max(len(examples), 1), 4.8))
for ax, r in zip(np.atleast_1d(axes), examples):
    img, m = load_img(r["id"]), load_mask(r["id"])
    S, _ = edge_score_map(gradient_magnitude(img), mask_boundary(m), SHIFT_R)
    im = ax.imshow(S, extent=(-SHIFT_R - 0.5, SHIFT_R + 0.5, SHIFT_R + 0.5, -SHIFT_R - 0.5), cmap="magma")
    ax.plot(0, 0, "w+", ms=12); ax.plot(r["dx"], r["dy"], "c*", ms=12)
    ax.set_title(f"{r['id']}\nconf={r['conf']:.1f} gain={r['gain']:.2f} best=({r['dx']:+d},{r['dy']:+d})", fontsize=8)
    ax.set_xlabel("dx (px)"); ax.set_ylabel("dy (px)")
    plt.colorbar(im, ax=ax, label="S = media de gradiente")
save_fig("b2_score_maps")

**Cómo leer este resultado:** `+` blanco = desplazamiento cero; `*` cian = mejor desplazamiento. El mapa de mayor confianza (izquierda) tiene un máximo claro; el de menor confianza (derecha) es casi plano y no debería usarse. Una `ganancia` de, por ejemplo, 1.3 significa que el borde desplazado cae sobre gradientes 30 % más intensos que el original.

## B3. Correlación cruzada de road-likeness (segundo proxy, independiente)
`R = (1 − S)·exp(−((V − 0.45)/0.25)²)` en HSV: favorece píxeles poco saturados (grises) de brillo medio. Es **tosco**: veredas, techos de calamina/concreto y suelo desnudo también son grises. Por eso no se usa solo, sino como contraste: si el offset por bordes y el offset por color coinciden en muchos parches, es difícil que ambos estén equivocados exactamente igual (cada uno depende de señales distintas: bordes vs apariencia interior).

In [ ]:
both = ok.dropna(subset=["ncc_dx", "ncc_dy"]).copy()
both["d_agree"] = np.hypot(both["dx"] - both["ncc_dx"], both["dy"] - both["ncc_dy"])
print(f"Acuerdo (distancia entre offsets <= 3 px): {100 * (both.d_agree <= 3).mean():.1f} % de {len(both)} parches")
print(f"Acuerdo exacto (<= 1 px): {100 * (both.d_agree <= 1).mean():.1f} %")
print(f"Mediana NCC offset: dx = {both['ncc_dx'].median():+.1f}, dy = {both['ncc_dy'].median():+.1f} px")
print(f"Correlación de dx entre métodos: {both['dx'].corr(both['ncc_dx']):.2f} | dy: {both['dy'].corr(both['ncc_dy']):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, a, b, name in [(axes[0], "dx", "ncc_dx", "dx"), (axes[1], "dy", "ncc_dy", "dy")]:
    ax.scatter(both[a], both[b], s=12, alpha=0.5)
    lim = SHIFT_R + 1
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.7)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel(f"{name} por bordes (px)"); ax.set_ylabel(f"{name} por road-likeness NCC (px)")
    ax.set_title(f"Acuerdo en {name}")
save_fig("b3_method_agreement")

**Cómo leer este resultado:** los puntos sobre la diagonal indican que ambos métodos estiman el mismo offset. Un acuerdo alto (> 60–70 % a ≤ 3 px) respalda los números del método principal; un acuerdo bajo no prueba que el método de bordes esté mal (el proxy de color es débil), pero obliga a apoyarse más en el QA manual.

## B4. Curva de sensibilidad IoU vs desplazamiento
Para una vía recta de ancho `w` desplazada `d` px de forma perpendicular, `IoU = (w − d)/(w + d)` (si `d < w`). Con `w = 7 px`, `d = 2` ya da IoU ≈ 0.56 y `d = 4` ≈ 0.27. Las calles angostas son muy sensibles al registro. Esta curva es un **techo** del IoU alcanzable con estas etiquetas y justifica reportar también la métrica relajada (tolerancia ρ) en el notebook 02.

In [ ]:
iou_ids = audit_ids[:min(100, len(audit_ids))]
iou_masks = [load_mask(p) for p in iou_ids]
iou_curve_ov = iou_sensitivity(iou_masks)
ds = np.arange(IOU_MAX_SHIFT + 1)
theory = [max(0.0, (7 - d) / (7 + d)) for d in ds]

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(ds, iou_curve_ov, "o-", label="Máscaras Overture (promedio)")
ax.plot(ds, theory, "k--", lw=0.9, label="Teórico: franja de 7 px")
ax.set_xlabel("Desplazamiento d (px)"); ax.set_ylabel("IoU(máscara, máscara desplazada)")
ax.set_title("Costo en IoU de un error de registro de d px"); ax.grid(alpha=0.3); ax.legend()
save_fig("b4_iou_vs_shift")
print({int(d): round(v, 3) for d, v in zip(ds, iou_curve_ov)})

**Cómo leer este resultado:** lee el IoU en el `d` que corresponde al offset mediano de B2. Si, por ejemplo, el offset típico es 3 px y la curva marca IoU ≈ 0.45, ningún modelo podrá superar ~0.45 de IoU "estricto" contra estas etiquetas aunque prediga bien la calle real. La curva observada se sitúa por encima de la teórica porque hay calles anchas y curvas.

## B5. Proxy de completitud (indicador grueso)
Fracción de píxeles con alta road-likeness (> `LIKE_THR`) que quedan fuera de la máscara dilatada `LIKE_DILATE_PX` px. Un valor alto **podría** indicar calles sin pavimentar o informales ausentes en Overture, pero también veredas, techos y suelo gris. Úsese solo de forma comparativa (Overture vs Massachusetts) y para elegir qué parches revisar a mano.

In [ ]:
miss = audit_df.dropna(subset=["missing_frac"])
print(f"Fracción mediana de 'road-likeness' alta fuera de la máscara: {100 * miss['missing_frac'].median():.1f} % (n={len(miss)})")

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.hist(100 * miss["missing_frac"], bins=30, color="#C44E52")
ax.set_xlabel("% de píxeles 'tipo calzada' fuera de la máscara"); ax.set_ylabel("n.º de parches")
ax.set_title("Proxy de calles faltantes (solo indicativo)")
save_fig("b5_missing_proxy")

top_missing = miss.sort_values("missing_frac").tail(4)["id"].tolist()
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, pid in zip(axes, top_missing):
    draw_contour(ax, load_img(pid), load_mask(pid)); ax.set_title(pid, fontsize=7)
fig.suptitle("Parches con mayor fracción fuera de la máscara: revisar a mano", y=1.02)
save_fig("b5_missing_examples")

## B6. Corrección opcional del desplazamiento
Si `APPLY_OFFSET_CORRECTION = True`, se escribe `masks_aligned/` desplazando cada parche confiable por su offset estimado y los demás por el offset global (mediana). **Riesgo:** el offset por parche es una estimación ruidosa basada en un proxy de bordes; corregir con ella puede alinear la máscara con sombras, bordes de techos o veredas y empeorar la etiqueta. Se recomienda corregir solo si el sesgo global es claro (nube compacta lejos de (0, 0)) y comparar entrenamientos con y sin corrección. Por defecto está desactivado.

In [ ]:
def write_aligned_masks(meta_df, ok_df):
    """Escribe masks_aligned/: offset por parche si es confiable, si no la mediana global."""
    os.makedirs(MASK_ALIGNED_DIR, exist_ok=True)
    gdx, gdy = int(round(ok_df["dx"].median())), int(round(ok_df["dy"].median()))
    per_patch = {r.id: (int(r.dx), int(r.dy)) for r in ok_df.itertuples()}
    for pid in tqdm(meta_df["id"], desc="masks_aligned"):
        dx, dy = per_patch.get(pid, (gdx, gdy))
        m = shift_mask(load_mask(pid), dx, dy)
        Image.fromarray(m * 255).save(f"{MASK_ALIGNED_DIR}/{pid}.png")
    print(f"Máscaras alineadas escritas en {MASK_ALIGNED_DIR} (offset global de respaldo: dx={gdx}, dy={gdy})")


if APPLY_OFFSET_CORRECTION and len(ok):
    write_aligned_masks(meta, ok)
else:
    print("APPLY_OFFSET_CORRECTION = False: no se generan máscaras corregidas.")

## B7. Plantilla de QA manual
Se sortean `N_MANUAL_QA` parches y se guarda (a) una hoja de contacto con el contorno de la máscara y (b) `manual_qa.csv` con columnas vacías para que el equipo las complete: `aligned_ok` (sí/no), `missing_roads` (sí/no), `extra_roads` (sí/no), `notes`. Es la validación independiente de los proxies automáticos.

In [ ]:
qa_sample = meta.sample(n=min(N_MANUAL_QA, len(meta)), random_state=SEED + 7)["id"].tolist()
pd.DataFrame({"id": qa_sample, "aligned_ok": "", "missing_roads": "", "extra_roads": "", "notes": ""}) \
    .to_csv(f"{RESULTS_DIR}/manual_qa.csv", index=False)

ncols = 6
nrows = math.ceil(len(qa_sample) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
for ax in np.atleast_1d(axes).ravel():
    ax.axis("off")
for ax, pid in zip(np.atleast_1d(axes).ravel(), qa_sample):
    draw_contour(ax, load_img(pid), load_mask(pid), lw=0.6)
    ax.set_title(pid, fontsize=6)
fig.suptitle("Hoja de contacto para QA manual (rellenar manual_qa.csv)", y=1.0)
save_fig("b7_manual_qa_contact_sheet")
print("Escrito:", f"{RESULTS_DIR}/manual_qa.csv")

## B8. Resumen de métricas de alineación (Overture)
Se consolidan todas las métricas en una tabla y se guardan en `results/alignment_overture.json` para enviarlas al equipo.

In [ ]:
def to_jsonable(o):
    if isinstance(o, dict):
        return {k: to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if (isinstance(o, float) and math.isnan(o)) or (isinstance(o, np.floating) and np.isnan(o)) else float(o)
    return o


summary_ov["iou_curve"] = {str(int(d)): v for d, v in zip(ds, iou_curve_ov)}
summary_ov["iou_at_2px"], summary_ov["iou_at_4px"] = iou_curve_ov[2], iou_curve_ov[4]
summary_ov["config"] = {"bbox": BBOX, "zoom": ZOOM, "road_width_mode": ROAD_WIDTH_MODE, "shift_radius_px": SHIFT_R,
                        "conf_min": CONF_MIN, "n_audit_patches": len(audit_ids), "seed": SEED,
                        "mean_m_per_px": float(meta["m_per_px"].mean())}
with open(f"{RESULTS_DIR}/alignment_overture.json", "w") as f:
    json.dump(to_jsonable(summary_ov), f, indent=2)
display(pd.Series({k: v for k, v in summary_ov.items() if not isinstance(v, (dict, list))}).to_frame("Overture"))
print("Escrito:", f"{RESULTS_DIR}/alignment_overture.json")

# PARTE C · Massachusetts Roads (benchmark de referencia)

Massachusetts Roads (Mnih, 2013) tiene imágenes aéreas de ~1 m/px (tiles de 1500×1500 px) y máscaras generadas rasterizando líneas centrales de OpenStreetMap con un ancho fijo. Es el benchmark clásico para comparar con la literatura (mIoU ≈ 59–63 %, F1 ≈ 75–79 % en baselines convolucionales). **Sus etiquetas también tienen ruido de registro** (OSM sobre aéreo), por eso se les aplica exactamente la misma auditoría, para comparar manzanas con manzanas.

## C1. Descubrimiento del dataset en `/kaggle/input`
Se busca por *glob* un directorio que contenga `train`, `train_labels`, `val`, `val_labels`, `test`, `test_labels`, sin depender del nombre exacto del dataset. Imagen y etiqueta se emparejan por nombre de archivo (sin extensión).

In [ ]:
def find_massachusetts():
    """Localiza la raíz que contiene train/val/test y sus *_labels bajo /kaggle/input."""
    for c in glob.glob("/kaggle/input/**/train_labels", recursive=True):
        root = os.path.dirname(c)
        if all(os.path.isdir(f"{root}/{d}") for d in ["train", "val", "test", "train_labels", "val_labels", "test_labels"]):
            return root
    raise FileNotFoundError(
        "No se encontró Massachusetts Roads en /kaggle/input. Agrega el dataset como input "
        "(Add Input -> buscar 'Massachusetts Roads Dataset', p. ej. balraj98/massachusetts-roads-dataset) "
        "y vuelve a ejecutar esta celda.")


def list_pairs(root, split):
    """Pares (imagen, etiqueta) de un split, emparejados por nombre de archivo sin extensión."""
    exts = (".tif", ".tiff", ".png")
    imgs = {os.path.splitext(f)[0]: f"{root}/{split}/{f}" for f in os.listdir(f"{root}/{split}") if f.lower().endswith(exts)}
    lbls = {os.path.splitext(f)[0]: f"{root}/{split}_labels/{f}" for f in os.listdir(f"{root}/{split}_labels") if f.lower().endswith(exts)}
    return [(k, imgs[k], lbls[k]) for k in sorted(set(imgs) & set(lbls))]


def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def read_label(path):
    return (np.array(Image.open(path).convert("L")) > 127).astype(np.uint8)


def blank_fraction(rgb):
    """Fracción de píxeles casi blancos (regiones sin datos del mosaico)."""
    return float((rgb.min(axis=2) >= 250).mean())


MASS_ROOT = None
try:
    MASS_ROOT = find_massachusetts()
    MASS_PAIRS = {s: list_pairs(MASS_ROOT, s) for s in ["train", "val", "test"]}
    print("Raíz:", MASS_ROOT)
    display(pd.Series({s: len(p) for s, p in MASS_PAIRS.items()}).to_frame("n_tiles"))
except FileNotFoundError as exc:
    print("AVISO (Parte C omitida hasta agregar el dataset):", exc)

## C2. EDA de Massachusetts
Se muestrean `MASS_N_EDA_TILES` tiles de entrenamiento para estimar: tamaño de tile, porcentaje de región en blanco (sin datos) y fracción de píxeles de vía.

In [ ]:
def mass_eda(pairs, n, seed):
    rng = np.random.default_rng(seed)
    pick = [pairs[i] for i in rng.choice(len(pairs), size=min(n, len(pairs)), replace=False)]
    rows = []
    for name, ip, lp in tqdm(pick, desc="EDA Massachusetts"):
        rgb, lab = read_rgb(ip), read_label(lp)
        rows.append({"id": name, "h": rgb.shape[0], "w": rgb.shape[1], "blank_frac": blank_fraction(rgb),
                     "road_ratio": float(lab.mean())})
    return pd.DataFrame(rows), pick


if MASS_ROOT:
    mass_eda_df, mass_pick = mass_eda(MASS_PAIRS["train"], MASS_N_EDA_TILES, SEED)
    print("Tamaños de tile:", mass_eda_df.groupby(["h", "w"]).size().to_dict())
    print(f"Blanco medio: {100 * mass_eda_df.blank_frac.mean():.2f} % | tiles con >{int(100 * MASS_BLANK_THR)} % en blanco: "
          f"{100 * (mass_eda_df.blank_frac > MASS_BLANK_THR).mean():.1f} %")
    print(f"Fracción de vía: media {100 * mass_eda_df.road_ratio.mean():.2f} % | mediana {100 * mass_eda_df.road_ratio.median():.2f} %")

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].hist(100 * mass_eda_df.road_ratio, bins=25, color="#4C72B0"); axes[0].set_xlabel("% de vía por tile")
    axes[1].hist(100 * mass_eda_df.blank_frac, bins=25, color="#8172B2"); axes[1].set_xlabel("% en blanco (sin datos) por tile")
    for ax in axes:
        ax.set_ylabel("n.º de tiles")
    save_fig("c2_mass_hist")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (name, ip, lp) in zip(axes, mass_pick[:3]):
        overlay_mask(ax, read_rgb(ip)[::3, ::3], read_label(lp)[::3, ::3]); ax.set_title(name, fontsize=8)
    save_fig("c2_mass_overlays")

**Cómo leer este resultado:** la fracción de vía en Massachusetts suele ser un orden de magnitud menor que 100 % y comparable (o algo menor) que en el dataset de Lima: es la evidencia de desbalance que justifica BCE + Dice. Los tiles con mucha región en blanco (mosaico sin datos) no deben usarse para entrenar; por eso el notebook 02 rechaza recortes con más de `MASS_BLANK_THR` de píxeles blancos.

**Estrategia de parches para el notebook 02:** los tiles de 1500×1500 no se pre-recortan a disco. Entrenamiento: recortes aleatorios de 512×512 al vuelo (descartando los que tengan > 25 % en blanco). Validación/test: rejilla determinista 3×3 de recortes de 512 (inicios 0, 494, 988; con solapamiento en los bordes), también descartando recortes casi vacíos. Así el conjunto de test es fijo y reproducible.

## C3. Misma auditoría de alineación sobre ~200 recortes de 512×512
Se reutilizan exactamente las funciones de la Parte B (`audit_patch`, `summarize_audit`, `iou_sensitivity`) sobre recortes aleatorios de los tiles de entrenamiento que tengan poco blanco y algo de vía.

In [ ]:
def sample_mass_crops(pairs, n, seed, size=PATCH):
    """Recortes aleatorios size x size con poco blanco y fracción de vía mínima. Devuelve [(id, rgb, mask)]."""
    rng = np.random.default_rng(seed)
    crops, tries = [], 0
    pbar = tqdm(total=n, desc="Recortes Massachusetts")
    while len(crops) < n and tries < n * 30:
        tries += 1
        name, ip, lp = pairs[int(rng.integers(len(pairs)))]
        rgb, lab = read_rgb(ip), read_label(lp)
        H, W = lab.shape
        for _ in range(8):
            y, x = int(rng.integers(0, H - size + 1)), int(rng.integers(0, W - size + 1))
            c_rgb, c_lab = rgb[y:y + size, x:x + size], lab[y:y + size, x:x + size]
            if blank_fraction(c_rgb) <= MASS_BLANK_THR and c_lab.mean() >= 0.005:
                crops.append((f"{name}_y{y}_x{x}", c_rgb, c_lab)); pbar.update(1)
                break
    pbar.close()
    return crops


if MASS_ROOT:
    mass_crops = sample_mass_crops(MASS_PAIRS["train"], MASS_N_AUDIT_CROPS, SEED)
    mass_df = run_audit(mass_crops, total=len(mass_crops))
    mass_df["m_per_px"] = MASS_M_PER_PX
    mass_df["road_ratio"] = [float(m.mean()) for _, _, m in mass_crops][:len(mass_df)]
    summary_ma = summarize_audit(mass_df, "massachusetts")
    iou_curve_ma = iou_sensitivity([m for _, _, m in mass_crops[:100]])
    summary_ma["iou_curve"] = {str(d): v for d, v in enumerate(iou_curve_ma)}
    summary_ma["iou_at_2px"], summary_ma["iou_at_4px"] = iou_curve_ma[2], iou_curve_ma[4]
    summary_ma["config"] = {"n_crops": len(mass_crops), "shift_radius_px": SHIFT_R, "conf_min": CONF_MIN,
                            "m_per_px_assumed": MASS_M_PER_PX, "seed": SEED}
    with open(f"{RESULTS_DIR}/alignment_massachusetts.json", "w") as f:
        json.dump(to_jsonable(summary_ma), f, indent=2)
    display(pd.Series({k: v for k, v in summary_ma.items() if not isinstance(v, (dict, list))}).to_frame("Massachusetts"))

    okm = mass_df[mass_df["valid"] & (mass_df["conf"] >= CONF_MIN)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    hb = axes[0].hexbin(okm["dx"], okm["dy"], gridsize=2 * SHIFT_R + 1, extent=(-SHIFT_R, SHIFT_R, -SHIFT_R, SHIFT_R), cmap="viridis", mincnt=1)
    axes[0].invert_yaxis(); axes[0].set_xlabel("dx (px)"); axes[0].set_ylabel("dy (px)"); axes[0].set_title("Offsets Massachusetts")
    plt.colorbar(hb, ax=axes[0])
    axes[1].plot(range(IOU_MAX_SHIFT + 1), iou_curve_ov, "o-", label="Overture (Lima)")
    axes[1].plot(range(IOU_MAX_SHIFT + 1), iou_curve_ma, "s-", label="Massachusetts")
    axes[1].set_xlabel("d (px)"); axes[1].set_ylabel("IoU con máscara desplazada"); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[1].set_title("Sensibilidad IoU vs desplazamiento")
    save_fig("c3_mass_alignment")

**Cómo leer este resultado:** si Massachusetts muestra offsets pequeños y compactos, sirve como referencia "limpia"; si también tiene dispersión, el ruido de registro es propio de las etiquetas basadas en OSM y no exclusivo de Overture. La curva IoU depende además del ancho de la vía (Massachusetts usa ancho fijo y estrecho, ~7 px; Overture usa ancho por clase).

## C4. Comparación final Overture vs Massachusetts

In [ ]:
if MASS_ROOT:
    keys = ["n_patches", "n_confident", "mean_road_ratio", "median_dx_px", "median_dy_px", "robust_std_dx_px", "robust_std_dy_px",
            "median_abs_offset_px", "median_abs_offset_m", "pct_le_2px", "pct_le_4px", "pct_gt_6px",
            "ncc_agreement_pct_le_3px", "median_missing_frac", "iou_at_2px", "iou_at_4px"]
    cmp_df = pd.DataFrame({"Overture (Lima)": {k: summary_ov.get(k) for k in keys},
                           "Massachusetts": {k: summary_ma.get(k) for k in keys}})
    display(cmp_df.round(3))
    cmp_df.to_csv(f"{RESULTS_DIR}/alignment_comparison.csv")
else:
    print("Sin Massachusetts: solo se dispone del resumen de Overture (results/alignment_overture.json).")

## Conclusiones y qué reportar
Completar con los números de las tablas anteriores después de ejecutar en Kaggle:

- **Cobertura:** `{N}` parches Overture (`{km}` km de vías conducibles); `{pct_zero}` % de parches sin vía; fracción media de vía `{x}` % (desbalance fondo:vía ≈ `{r}`:1) → se justifica BCE + Dice.
- **Sesgo sistemático Overture:** mediana (dx, dy) = (`{dx}`, `{dy}`) px ≈ (`{dx_m}`, `{dy_m}`) m. ¿Es corregible por traslación global? `{sí/no}`.
- **Error aleatorio Overture:** desviación robusta `{std}` px; `{p2}` % de parches con |offset| ≤ 2 px, `{p4}` % ≤ 4 px, `{p6}` % > 6 px.
- **Acuerdo entre proxies** (bordes vs road-likeness): `{agree}` %. Resultado del QA manual (30 parches): `{k}` de 30 alineados.
- **Techo de IoU por registro:** IoU a 2 px = `{iou2}`, a 4 px = `{iou4}` → justifica la métrica relajada (ρ = 3 px) en el notebook 02.
- **Comparación con Massachusetts:** `{más/menos}` ruidoso que Overture en offset mediano (`{a}` vs `{b}` px); fracción de vía `{x}` % vs `{y}` %.
- **Riesgos para el informe:** vías informales/sin pavimentar probablemente subrepresentadas en Overture (sesgo hacia vías formales); términos de uso de Esri por verificar; resolución ~1.17 m/px limita calles angostas.
- **Decisión sobre la corrección de offset:** `{se aplica / no se aplica}` porque `{motivo}`.

## Cómo guardar la salida como Kaggle Dataset (para el notebook 02)
1. Ejecutar el notebook completo con **Save Version → Save & Run All (Commit)**.
2. Abrir la versión guardada → pestaña **Output** → botón **New Dataset**; incluir la carpeta `overture_lima/` (con `images/`, `masks/`, `patches.csv` y, si se generó, `masks_aligned/`). Nombrarlo, por ejemplo, `overture-lima-patches` y dejarlo **privado** hasta verificar los términos de Esri.
3. En el notebook 02: **Add Input** → el dataset recién creado. El notebook lo descubre automáticamente buscando `patches.csv` bajo `/kaggle/input`.
4. Descargar también `results/*.json` y `figures/` para el informe.